In [ ]:
# Cell 0 — Imports
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import umap
import pickle
import warnings
warnings.filterwarnings('ignore')

BASE        = Path('../../Models/PickleFiles/NewModel')
GRADES_PATH = Path('../../Models/PickleFiles/AVgrades.pkl')
OUT         = BASE
K_COMPS     = 5
PEAK_AGE    = {'QB': 30, 'RB': 25, 'WR': 26, 'TE': 27}

In [ ]:
# Cell 1 — Load team grades and prep slim version
grades_raw = pd.read_pickle(GRADES_PATH)
grades = grades_raw[['season', 'team', 'rb', 'oline', 'wrte', 'qb']].copy()
grades = grades.rename(columns={
    'rb':    'rb_grade',
    'oline': 'oline_grade',
    'wrte':  'wrte_grade',
    'qb':    'qb_grade',
})
print(f'Team grades: {grades.shape}, seasons {sorted(grades["season"].unique())}')
print(grades[grades['season']==2025][['team','rb_grade','wrte_grade','qb_grade','oline_grade']].head())

In [ ]:
# Cell 2 — Load full datasets and add new_team flag

def add_new_team_flag(df):
    df = df.sort_values(['player_name', 'season']).copy()
    df['prev_team'] = df.groupby('player_name')['team'].shift(1)
    df['new_team'] = ((df['prev_team'].notna()) & (df['team'] != df['prev_team'])).astype(int)
    return df.drop(columns=['prev_team'])

pos_dfs = {}
for pos in ['rb', 'wr', 'te', 'qb']:
    df = pd.read_pickle(BASE / f'{pos}_full.pkl')
    df = add_new_team_flag(df)
    pos_dfs[pos] = df
    print(f'{pos.upper()}: {df.shape}, seasons {sorted(df["season"].unique())}')

In [ ]:
# Cell 3 — Define entering-season feature sets
# All features describe the player's situation ENTERING a season:
#   - lag columns  = prior season performance / usage
#   - age/seasons  = what they are ENTERING this season
#   - team grades  = the team they are walking into

FEATURES = {
    'rb': ['age', 'seasons_played', 'age_curve', 'new_team',
           'ppg_lag1', 'weighted_ppg',
           'lag1_carries_pg', 'lag1_targets_pg', 'lag1_target_share',
           'rb_grade', 'oline_grade'],
    'wr': ['age', 'seasons_played', 'age_curve', 'new_team',
           'ppg_lag1', 'weighted_ppg',
           'lag1_targets_pg', 'lag1_target_share', 'lag1_wopr',
           'wrte_grade'],
    'te': ['age', 'seasons_played', 'age_curve', 'new_team',
           'ppg_lag1', 'weighted_ppg',
           'lag1_targets_pg', 'lag1_target_share',
           'wrte_grade'],
    'qb': ['age', 'seasons_played', 'age_curve', 'new_team',
           'ppg_lag1', 'weighted_ppg',
           'lag1_attempts_pg', 'lag1_comp_pct',
           'qb_grade'],
}

RADAR_FEATURES = {
    'rb': ['age', 'seasons_played', 'ppg_lag1', 'lag1_carries_pg', 'lag1_target_share', 'new_team', 'rb_grade', 'oline_grade'],
    'wr': ['age', 'seasons_played', 'ppg_lag1', 'lag1_targets_pg', 'lag1_target_share', 'lag1_wopr', 'new_team', 'wrte_grade'],
    'te': ['age', 'seasons_played', 'ppg_lag1', 'lag1_targets_pg', 'lag1_target_share', 'new_team', 'wrte_grade'],
    'qb': ['age', 'seasons_played', 'ppg_lag1', 'lag1_attempts_pg', 'lag1_comp_pct', 'new_team', 'qb_grade'],
}

print('Feature counts:', {k: len(v) for k, v in FEATURES.items()})

In [ ]:
# Cell 4 — Build historical and current (entering-2026) datasets

def build_entering_profile(pos, df, grades):
    """
    Historical: seasons where ppg_lag1 is available (2021+).
    Each row = a player's situation ENTERING that season.
    Outcome = ppg (what they did that season).

    Current (entering 2026): 2025 season data, age/seasons shifted +1,
    2025 actual ppg/usage become the lag (prior season).
    """
    pos_upper = pos.upper()
    peak = PEAK_AGE.get(pos_upper, 28)
    grade_cols = {'rb': ['rb_grade','oline_grade'], 'wr': ['wrte_grade'],
                  'te': ['wrte_grade'], 'qb': ['qb_grade']}

    # ── Historical: entering seasons 2021-2025 ──
    hist = df[df['season'] >= 2021].copy()
    # Join team grades for the season being entered
    hist = hist.merge(grades[['season','team'] + grade_cols[pos]], on=['season','team'], how='left')
    hist = hist.dropna(subset=['ppg_lag1'])  # need prior season data
    hist = hist.dropna(subset=FEATURES[pos])
    hist['outcome_ppg'] = hist['ppg']  # what they did that season
    hist['is_current']  = False

    # ── Current: entering 2026 ──
    curr = df[df['season'] == 2025].copy()
    # Shift age/experience forward by 1
    curr['age']          = curr['age'] + 1
    curr['seasons_played'] = curr['seasons_played'] + 1
    curr['age_curve']    = peak - curr['age']  # recalculate at entering age
    # 2025 actuals become the prior-season lags
    curr['ppg_lag1']     = curr['ppg']
    curr['weighted_ppg'] = curr['weighted_ppg']  # already trend-based, keep as-is
    if 'targets_pg'  in curr.columns: curr['lag1_targets_pg']  = curr['targets_pg']
    if 'target_share' in curr.columns: curr['lag1_target_share'] = curr['target_share']
    if 'carries_pg'  in curr.columns: curr['lag1_carries_pg']  = curr['carries_pg']
    if 'wopr'        in curr.columns: curr['lag1_wopr']         = curr['wopr']
    if 'attempts_pg' in curr.columns: curr['lag1_attempts_pg']  = curr['attempts_pg']
    if 'comp_pct'    in curr.columns: curr['lag1_comp_pct']     = curr['comp_pct']
    # Join 2025 team grades (best available proxy for their 2026 team quality)
    curr = curr.merge(grades[grades['season']==2025][['team'] + grade_cols[pos]], on='team', how='left')
    curr = curr.dropna(subset=FEATURES[pos])
    curr['outcome_ppg'] = np.nan  # unknown — what we want to find comps for
    curr['is_current']  = True

    print(f'{pos_upper}: {len(hist)} historical entering-season rows, {len(curr)} entering-2026 players')
    return hist.reset_index(drop=True), curr.reset_index(drop=True)

historical = {}
current    = {}
for pos in ['rb', 'wr', 'te', 'qb']:
    historical[pos], current[pos] = build_entering_profile(pos, pos_dfs[pos], grades)

In [ ]:
# Cell 5 — Fit KNN and find comps for each 2026 player

all_comps  = {}
all_scaled = {}

for pos in ['rb', 'wr', 'te', 'qb']:
    hist = historical[pos]
    curr = current[pos]
    feat_cols   = FEATURES[pos]
    radar_cols  = RADAR_FEATURES[pos]

    scaler      = StandardScaler()
    hist_scaled = scaler.fit_transform(hist[feat_cols])
    curr_scaled = scaler.transform(curr[feat_cols])

    nn = NearestNeighbors(n_neighbors=K_COMPS, metric='euclidean')
    nn.fit(hist_scaled)
    distances, indices = nn.kneighbors(curr_scaled)

    for i, row in curr.iterrows():
        player = row['player_name']
        comps  = []
        for dist, idx in zip(distances[i], indices[i]):
            comp = hist.iloc[idx]
            comp_feat = {c: round(float(comp[c]), 2) for c in radar_cols if c in comp.index}
            curr_feat = {c: round(float(row[c]),  2) for c in radar_cols if c in row.index}
            comps.append({
                'player':        comp['player_name'],
                'season':        int(comp['season']),
                'team':          str(comp['team']),
                'outcome_ppg':   round(float(comp['outcome_ppg']), 2),
                'similarity':    round(1 / (1 + dist), 3),
                'comp_features': comp_feat,
                'curr_features': curr_feat,
            })
        all_comps[player] = comps

    all_scaled[pos] = {
        'hist_df':     hist,
        'curr_df':     curr,
        'hist_scaled': hist_scaled,
        'curr_scaled': curr_scaled,
    }
    print(f'{pos.upper()}: comps built for {len(curr)} players')

print(f'\nTotal players with comps: {len(all_comps)}')

In [ ]:
# Cell 6 — Fit UMAP per position on historical + current combined

umap_records = []

for pos, data in all_scaled.items():
    hist_df     = data['hist_df']
    curr_df     = data['curr_df']
    hist_scaled = data['hist_scaled']
    curr_scaled = data['curr_scaled']

    combined = np.vstack([hist_scaled, curr_scaled])
    reducer  = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
    coords   = reducer.fit_transform(combined)

    n_hist = len(hist_df)

    for i, (x, y) in enumerate(coords[:n_hist]):
        r = hist_df.iloc[i]
        umap_records.append({
            'player_name': r['player_name'],
            'season':      int(r['season']),
            'team':        str(r['team']),
            'position':    pos.upper(),
            'age':         float(r['age']),
            'ppg':         round(float(r['outcome_ppg']), 2),
            'ppg_lag1':    round(float(r['ppg_lag1']), 2),
            'is_current':  False,
            'x': round(float(x), 4),
            'y': round(float(y), 4),
        })

    for i, (x, y) in enumerate(coords[n_hist:]):
        r = curr_df.iloc[i]
        umap_records.append({
            'player_name': r['player_name'],
            'season':      2026,
            'team':        str(r['team']),
            'position':    pos.upper(),
            'age':         float(r['age']),
            'ppg':         round(float(r['ppg_lag1']), 2),  # prior season (2025 actual)
            'ppg_lag1':    round(float(r['ppg_lag1']), 2),
            'is_current':  True,
            'x': round(float(x), 4),
            'y': round(float(y), 4),
        })

    print(f'{pos.upper()} UMAP: {n_hist} hist + {len(curr_df)} current')

umap_df = pd.DataFrame(umap_records)
print(f'\nTotal UMAP points: {len(umap_df)} ({umap_df["is_current"].sum()} entering-2026)')

In [ ]:
# Cell 7 — Save and sanity check

with open(OUT / 'similarity_comps.pkl', 'wb') as f:
    pickle.dump(all_comps, f)
umap_df.to_pickle(OUT / 'umap_coords.pkl')

print('Saved similarity_comps.pkl and umap_coords.pkl')

# Sanity check a few players
import sys
sys.stdout.reconfigure(encoding='utf-8') if hasattr(sys.stdout, 'reconfigure') else None

for name in ['Wandale Robinson', 'Puka Nacua', 'Jahmyr Gibbs', 'Josh Allen']:
    if name in all_comps:
        print(f'\n{name} (entering 2026):')
        for c in all_comps[name]:
            print(f'  {c["player"]} entering {c["season"]} ({c["team"]}) -> did {c["outcome_ppg"]} PPG | sim={c["similarity"]}')